# 04 — Report Assets

Slide-native figures + PPTX deck assembly for the Olist portfolio.

**Deliverables (land in `reports/`):**
- 5 slide-native PNGs (`reports/figures/slide_*.png`)
- `reports/olist_deck.pptx` — 14-slide deck, flat/minimal theme

**Deck structure:**
1. Cover
2. Thesis
3. Headline — 30-day cliff
4. Headline — on-time operational lever
5. Headline — price & category don't move satisfaction
6. Headline — state is delivery-mediated
7. Methodology
8. Limitations
9. What I'd do next
10. Thanks / contact
11. Appendix A — full 9-test results table
12. Appendix B — office_furniture scatter
13. Appendix C — Q1 price→delivery
14. Appendix D — §1 category→price

**PDF export:** python-pptx writes the `.pptx`; open it in PowerPoint and use *File → Export → Create PDF/XPS* to produce the PDF. LibreOffice users can run `soffice --headless --convert-to pdf olist_deck.pptx` from the terminal.

## Part 1 — Figures

### Cell 1 — Setup, theme, data load

In [1]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
FIGURES_DIR = PROJECT_ROOT / 'reports' / 'figures'
DATA_PATH = PROJECT_ROOT / 'data' / 'processed' / 'orders_master.parquet'

assert DATA_PATH.exists(), f"Data not found at {DATA_PATH}"
assert FIGURES_DIR.exists(), f"Figures dir missing: {FIGURES_DIR}"

df = pd.read_parquet(DATA_PATH)
print(f"Loaded {len(df):,} orders, {df.shape[1]} columns")
print(f"Kernel: {sys.executable}")

# Flat/minimal theme — single accent, neutral grid, warning red for callouts
FONT         = "Arial, Helvetica, sans-serif"
ACCENT       = "#0B5FB4"
ACCENT_LIGHT = "#8FB4D9"
NEUTRAL      = "#333333"
GRID         = "#E5E5E5"
WARNING      = "#C84545"
MUTED        = "#AAAAAA"

def style_slide(fig, title, subtitle=None, width=1200, height=675, extra_top=0):
    t = title
    if subtitle:
        t = f"{title}<br><span style='font-size:14px;color:#777;font-weight:normal'>{subtitle}</span>"
    fig.update_layout(
        font=dict(family=FONT, color=NEUTRAL, size=14),
        title=dict(text=t, font=dict(size=22, color=NEUTRAL), x=0.03, xanchor='left', y=0.96),
        plot_bgcolor='white', paper_bgcolor='white',
        margin=dict(l=70, r=40, t=110 + extra_top, b=70),
        width=width, height=height,
    )
    fig.update_xaxes(showgrid=False, showline=True, linecolor=GRID, ticks='outside',
                     tickcolor=GRID, title_font_size=14)
    fig.update_yaxes(showgrid=True, gridcolor=GRID, showline=False, zeroline=False,
                     ticks='outside', tickcolor=GRID, title_font_size=14)
    return fig

def save_slide(fig, name):
    path = FIGURES_DIR / f"slide_{name}.png"
    fig.write_image(str(path), scale=2)
    print(f"Saved {path.name}")

Loaded 99,441 orders, 26 columns
Kernel: /home/lemfi/code/workintechpoyrazaka-sketch/olist-ecommerce-analysis/venv/bin/python


### Cell 2 — The 30-day cliff (slide 3)

In [2]:
delivered = df[df['order_status'] == 'delivered'].copy()
delivered = delivered[delivered['delivery_days'].notna() & delivered['review_score'].notna()]

bins = [-0.1, 7, 14, 21, 30, 60, float(delivered['delivery_days'].max()) + 1]
labels = ['0–7', '8–14', '15–21', '22–30', '31–60', '61+']
delivered['bin'] = pd.cut(delivered['delivery_days'], bins=bins, labels=labels)

cliff = (delivered.groupby('bin', observed=True)
         .agg(n=('review_score','size'),
              top=('review_score', lambda s: (s == 5).mean() * 100),
              bot=('review_score', lambda s: (s == 1).mean() * 100))
         .reset_index())
print(cliff)

fig = go.Figure()
fig.add_trace(go.Bar(
    x=cliff['bin'], y=cliff['top'], name='5★ (top-box)', marker_color=ACCENT,
    text=[f"{v:.1f}%" for v in cliff['top']], textposition='outside',
    textfont=dict(size=12, color=ACCENT),
))
fig.add_trace(go.Bar(
    x=cliff['bin'], y=cliff['bot'], name='1★ (bottom-box)', marker_color=WARNING,
    text=[f"{v:.1f}%" for v in cliff['bot']], textposition='outside',
    textfont=dict(size=12, color=WARNING),
))
fig.add_annotation(
    x='31–60', y=62,
    text="<b>The 30-day cliff</b><br>5★: 40.8% → 16.5%<br>1★: 18.7% → 54.1%",
    showarrow=True, arrowhead=2, arrowcolor=NEUTRAL, arrowsize=1, arrowwidth=1.5,
    ax=-100, ay=-55, align='left',
    font=dict(size=12, color=NEUTRAL),
    bgcolor='rgba(255,255,255,0.95)', bordercolor=MUTED, borderwidth=1, borderpad=8,
)
style_slide(fig,
    title="Satisfaction doesn't decay — it collapses at day 30",
    subtitle=f"n = {len(delivered):,} delivered orders, Jan 2017 – Aug 2018")
fig.update_layout(barmode='group', showlegend=True,
                  legend=dict(x=0.02, y=0.98, xanchor='left', yanchor='top',
                              bgcolor='rgba(255,255,255,0.9)', bordercolor=GRID, borderwidth=1))
fig.update_xaxes(title_text="Delivery days")
fig.update_yaxes(title_text="% of orders in bin", range=[0, 80])

fig.show()
save_slide(fig, "01_thirtyday_cliff")

     bin      n        top        bot
0    0–7  25933  68.376200   5.325261
1   8–14  39997  63.242243   6.495487
2  15–21  17582  55.568195   8.451826
3  22–30   7882  40.839888  18.675463
4  31–60   4139  16.501570  54.071032
5    61+    291  15.463918  60.824742


RuntimeError: 

Kaleido requires Google Chrome to be installed.

Either download and install Chrome yourself following Google's instructions for your operating system,
or install it from your terminal by running:

    $ plotly_get_chrome



### Cell 3 — On-time vs late distribution (slide 4)

In [ ]:
ot = delivered[delivered['on_time'].notna()].copy()
on_time_dist = ot[ot['on_time']]['review_score'].value_counts(normalize=True).sort_index() * 100
late_dist    = ot[~ot['on_time']]['review_score'].value_counts(normalize=True).sort_index() * 100
n_on, n_late = ot['on_time'].sum(), (~ot['on_time']).sum()
pct_late = n_late / len(ot) * 100

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=(f'On-time ({n_on:,} — {100-pct_late:.1f}%)',
                    f'Late ({n_late:,} — {pct_late:.1f}%)'),
    horizontal_spacing=0.14)

fig.add_trace(go.Bar(
    x=[f"{int(s)}★" for s in on_time_dist.index], y=on_time_dist.values,
    marker_color=ACCENT,
    text=[f"{v:.1f}%" for v in on_time_dist.values], textposition='outside',
    textfont=dict(size=12, color=ACCENT), showlegend=False,
), row=1, col=1)
fig.add_trace(go.Bar(
    x=[f"{int(s)}★" for s in late_dist.index], y=late_dist.values,
    marker_color=WARNING,
    text=[f"{v:.1f}%" for v in late_dist.values], textposition='outside',
    textfont=dict(size=12, color=WARNING), showlegend=False,
), row=1, col=2)

style_slide(fig,
    title="On-time delivery is the single operational lever for satisfaction",
    subtitle="OR = 12.2× for 1★ reviews  |  +39.6pp bottom-box gap  |  rank-biserial |r| = 0.55",
    extra_top=20)
fig.update_yaxes(title_text="% of orders", range=[0, 75], row=1, col=1)
fig.update_yaxes(range=[0, 75], row=1, col=2)
fig.update_xaxes(title_text="Review score", row=1, col=1)
fig.update_xaxes(title_text="Review score", row=1, col=2)

fig.show()
save_slide(fig, "02_ontime_vs_late")

Saved slide_02_ontime_vs_late.png


### Cell 4 — Thesis contrast two-panel (slide 5)

In [ ]:
MIN_N = 100
cat_stats = (delivered.groupby('category', observed=True)
             .agg(n=('review_score','size'),
                  median_price=('items_price_total','median'),
                  mean_review=('review_score','mean'))
             .reset_index()
             .query(f'n >= {MIN_N}'))
cat_stats['log_price'] = np.log10(cat_stats['median_price'])

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=(
        "Category median price (log₁₀ R$) — ω² = 0.129  <b>moderate</b>",
        "Category mean review score — η²_H = 0.008  <b>trivial</b>",
    ),
    horizontal_spacing=0.15)
fig.add_trace(go.Histogram(
    x=cat_stats['log_price'], nbinsx=18,
    marker_color=ACCENT, marker_line_color='white', marker_line_width=1,
    showlegend=False,
), row=1, col=1)
fig.add_trace(go.Histogram(
    x=cat_stats['mean_review'], nbinsx=18,
    marker_color=ACCENT_LIGHT, marker_line_color='white', marker_line_width=1,
    showlegend=False,
), row=1, col=2)

style_slide(fig,
    title="Categories disagree on what things cost, not on how satisfied customers are",
    subtitle=f"{len(cat_stats)} categories (n ≥ {MIN_N})  |  price ranges 55× across catalog  |  review means cluster around 4",
    width=1400, height=600, extra_top=20)
fig.update_xaxes(title_text="log₁₀(median price, R$)", row=1, col=1)
fig.update_xaxes(title_text="Mean review score", range=[1, 5], row=1, col=2)
fig.update_yaxes(title_text="# categories", row=1, col=1)
fig.update_yaxes(title_text="# categories", row=1, col=2)

fig.show()
save_slide(fig, "03_thesis_contrast")

Saved slide_03_thesis_contrast.png


### Cell 5 — office_furniture callout (appendix B)

In [ ]:
cat_full = (delivered.groupby('category', observed=True)
            .agg(n=('review_score','size'),
                 median_price=('items_price_total','median'),
                 bottom_box=('review_score', lambda s: (s == 1).mean() * 100))
            .reset_index()
            .query(f'n >= {MIN_N}'))
cat_full['is_of'] = cat_full['category'] == 'office_furniture'

fig = go.Figure()
other = cat_full[~cat_full['is_of']]
fig.add_trace(go.Scatter(
    x=other['median_price'], y=other['bottom_box'],
    mode='markers',
    marker=dict(size=np.log10(other['n']) * 5, color=ACCENT_LIGHT, opacity=0.6,
                line=dict(width=0.5, color='white')),
    text=other['category'],
    hovertemplate="%{text}<br>Median: R$%{x:.0f}<br>1★ rate: %{y:.1f}%<extra></extra>",
    showlegend=False,
))
of = cat_full[cat_full['is_of']]
fig.add_trace(go.Scatter(
    x=of['median_price'], y=of['bottom_box'],
    mode='markers+text',
    marker=dict(size=np.log10(of['n'].iloc[0]) * 6 + 4, color=WARNING,
                line=dict(width=2, color='white')),
    text=["  <b>office_furniture</b>"], textposition='middle right',
    textfont=dict(size=14, color=WARNING),
    hovertemplate="office_furniture<br>Median: R$%{x:.0f}<br>1★ rate: %{y:.1f}%<extra></extra>",
    showlegend=False,
))
fig.add_hline(y=9.8, line_dash="dot", line_color=MUTED, line_width=1,
              annotation_text="catalog avg 9.8%", annotation_position="top left",
              annotation_font=dict(size=11, color=MUTED))

style_slide(fig,
    title="office_furniture: expensive and unsatisfying — the only category combining both",
    subtitle="Top-5 median price AND bottom-5 satisfaction  |  n = 1,270 orders  |  17.2% 1★ rate vs 9.8% catalog average")
fig.update_xaxes(title_text="Category median price (R$, log scale)", type='log')
fig.update_yaxes(title_text="1★ (bottom-box) rate, %")

fig.show()
save_slide(fig, "04_office_furniture")

Saved slide_04_office_furniture.png


### Cell 6 — State delivery-mediation (slide 6)

In [ ]:
from scipy.stats import spearmanr

state_stats = (delivered.groupby('customer_state', observed=True)
               .agg(n=('review_score','size'),
                    mean_delivery=('delivery_days','mean'),
                    bottom_box=('review_score', lambda s: (s == 1).mean() * 100))
               .reset_index()
               .sort_values('n', ascending=False)
               .head(10))
print(state_stats)

rho, _ = spearmanr(state_stats['mean_delivery'], state_stats['bottom_box'])
print(f"Spearman ρ (top-10 states, delivery vs bottom-box): {rho:.3f}")

state_stats['is_rj'] = state_stats['customer_state'] == 'RJ'

fig = go.Figure()
other = state_stats[~state_stats['is_rj']]
fig.add_trace(go.Scatter(
    x=other['mean_delivery'], y=other['bottom_box'],
    mode='markers+text',
    marker=dict(size=np.log10(other['n']) * 8, color=ACCENT, opacity=0.75,
                line=dict(width=1, color='white')),
    text=other['customer_state'], textposition='top center',
    textfont=dict(size=12, color=NEUTRAL),
    hovertemplate="%{text}<br>Mean delivery: %{x:.1f} days<br>1★ rate: %{y:.1f}%<extra></extra>",
    showlegend=False,
))
rj = state_stats[state_stats['is_rj']]
fig.add_trace(go.Scatter(
    x=rj['mean_delivery'], y=rj['bottom_box'],
    mode='markers+text',
    marker=dict(size=np.log10(rj['n'].iloc[0]) * 9, color=WARNING,
                line=dict(width=2, color='white')),
    text=["  <b>RJ</b>"], textposition='middle right',
    textfont=dict(size=14, color=WARNING),
    hovertemplate="RJ<br>Mean delivery: %{x:.1f} days<br>1★ rate: %{y:.1f}%<extra></extra>",
    showlegend=False,
))

style_slide(fig,
    title="State satisfaction is delivery-mediated, not regional",
    subtitle=f"Spearman ρ = {rho:.2f} across top 10 states  |  RJ: ~5pp higher 1★ rate than delivery time predicts")
fig.update_xaxes(title_text="Mean delivery time (days)")
fig.update_yaxes(title_text="1★ (bottom-box) rate, %")

fig.show()
save_slide(fig, "05_state_delivery_mediation")

   customer_state      n  mean_delivery  bottom_box
25             SP  40266       8.747180    7.912383
18             RJ  12211      15.241871   14.757186
10             MG  11285      11.983983    8.949934
22             RS   5326      15.280345    8.993616
17             PR   4900      11.972297    7.938776
23             SC   3519      14.879042    9.832339
4              BA   3229      19.245509   13.223908
6              DF   2070      12.961629   10.289855
7              ES   1969      15.609603   11.020823
8              GO   1946      15.558286    9.866393
Spearman ρ (top-10 states, delivery vs bottom-box): 0.758


Saved slide_05_state_delivery_mediation.png


### Cell 7 — Verify all 5 slide figures exported

In [2]:
expected = [
    "slide_01_thirtyday_cliff.png",
    "slide_02_ontime_vs_late.png",
    "slide_03_thesis_contrast.png",
    "slide_04_office_furniture.png",
    "slide_05_state_delivery_mediation.png",
]
print("Slide figures in", FIGURES_DIR)
all_ok = True
for name in expected:
    p = FIGURES_DIR / name
    status = "OK" if p.exists() else "MISSING"
    size = f"{p.stat().st_size / 1024:.0f} KB" if p.exists() else "—"
    print(f"  [{status}]  {name:<45}  {size}")
    all_ok = all_ok and p.exists()

# Verify analysis figures needed for appendix exist too
analysis_needed = [
    "07_q1_price_vs_delivery.png",
    "13_s1_category_vs_price.png",
]
print("\nAnalysis figures (for appendix):")
for name in analysis_needed:
    p = FIGURES_DIR / name
    status = "OK" if p.exists() else "MISSING"
    print(f"  [{status}]  {name}")

assert all_ok, "Some slide figures missing. Re-run Cells 2–6 above."
print("\nAll figures ready for deck assembly.")

Slide figures in /home/lemfi/code/workintechpoyrazaka-sketch/olist-ecommerce-analysis/reports/figures
  [OK]  slide_01_thirtyday_cliff.png                   180 KB
  [OK]  slide_02_ontime_vs_late.png                    176 KB
  [OK]  slide_03_thesis_contrast.png                   172 KB
  [OK]  slide_04_office_furniture.png                  198 KB
  [OK]  slide_05_state_delivery_mediation.png          156 KB

Analysis figures (for appendix):
  [OK]  07_q1_price_vs_delivery.png
  [OK]  13_s1_category_vs_price.png

All figures ready for deck assembly.


## Part 2 — Deck Assembly

Builds `reports/olist_deck.pptx` — 14 slides, 16:9 widescreen, flat/minimal theme.

Requires `python-pptx` and `Pillow` (both in `requirements.txt`).

### Cell 8 — PPTX imports, brand tokens, helpers

In [3]:
from PIL import Image
from pptx import Presentation
from pptx.util import Inches, Pt
from pptx.dml.color import RGBColor
from pptx.enum.text import PP_ALIGN, MSO_ANCHOR
from pptx.enum.shapes import MSO_SHAPE

OUT_PPTX = PROJECT_ROOT / 'reports' / 'olist_deck.pptx'

# Brand — matches figure colors
ACCENT   = RGBColor(0x0B, 0x5F, 0xB4)
WARNING  = RGBColor(0xC8, 0x45, 0x45)
NEUTRAL_ = RGBColor(0x33, 0x33, 0x33)
MUTED_   = RGBColor(0xAA, 0xAA, 0xAA)
SOFT     = RGBColor(0xF5, 0xF7, 0xFA)
WHITE    = RGBColor(0xFF, 0xFF, 0xFF)
PPT_FONT = "Arial"

# Author card — TODO: update LinkedIn / email before committing
AUTHOR       = "Poi"
GITHUB_URL   = "github.com/workintechpoyrazaka-sketch/olist-ecommerce-analysis"
LINKEDIN_URL = "linkedin.com/in/adilpoyrazaka"
EMAIL        = "akaadilpoyraz@gmail.com"
DATE_STR     = "April 2026"

SLIDE_W = Inches(13.333)
SLIDE_H = Inches(7.5)
MARGIN  = Inches(0.6)

ANCHOR_MAP = {"top": MSO_ANCHOR.TOP, "middle": MSO_ANCHOR.MIDDLE, "bottom": MSO_ANCHOR.BOTTOM}
ALIGN_MAP  = {"left": PP_ALIGN.LEFT, "center": PP_ALIGN.CENTER, "right": PP_ALIGN.RIGHT}

def _set_run(run, text, size=14, bold=False, color=None, italic=False, font=PPT_FONT):
    run.text = text
    run.font.name = font
    run.font.size = Pt(size)
    run.font.bold = bold
    run.font.italic = italic
    if color is not None:
        run.font.color.rgb = color

def add_text(slide, text, left, top, width, height,
             size=14, bold=False, color=NEUTRAL_, align="left",
             anchor="top", italic=False, font=PPT_FONT):
    tb = slide.shapes.add_textbox(left, top, width, height)
    tf = tb.text_frame
    tf.word_wrap = True
    tf.margin_left = tf.margin_right = Inches(0.05)
    tf.margin_top = tf.margin_bottom = Inches(0.02)
    tf.vertical_anchor = ANCHOR_MAP[anchor]
    p = tf.paragraphs[0]
    p.alignment = ALIGN_MAP[align]
    _set_run(p.add_run(), text, size=size, bold=bold, color=color, italic=italic, font=font)
    return tb

def add_multi_text(slide, blocks, left, top, width, height, align="left", anchor="top"):
    tb = slide.shapes.add_textbox(left, top, width, height)
    tf = tb.text_frame
    tf.word_wrap = True
    tf.margin_left = tf.margin_right = Inches(0.05)
    tf.margin_top = tf.margin_bottom = Inches(0.02)
    tf.vertical_anchor = ANCHOR_MAP[anchor]
    for i, b in enumerate(blocks):
        p = tf.paragraphs[0] if i == 0 else tf.add_paragraph()
        p.alignment = ALIGN_MAP.get(b.get("align", align), PP_ALIGN.LEFT)
        if "space_before" in b: p.space_before = Pt(b["space_before"])
        if "space_after"  in b: p.space_after  = Pt(b["space_after"])
        _set_run(p.add_run(), b["text"],
                 size=b.get("size", 14), bold=b.get("bold", False),
                 color=b.get("color", NEUTRAL_), italic=b.get("italic", False),
                 font=b.get("font", PPT_FONT))
    return tb

def add_panel(slide, left, top, width, height, fill=SOFT):
    shape = slide.shapes.add_shape(MSO_SHAPE.RECTANGLE, left, top, width, height)
    shape.fill.solid()
    shape.fill.fore_color.rgb = fill
    shape.line.fill.background()
    shape.shadow.inherit = False
    return shape

def add_footer(slide, page_num, total):
    add_text(slide, f"Olist Analysis  ·  {AUTHOR}  ·  {DATE_STR}",
             MARGIN, Inches(7.15), Inches(9), Inches(0.25),
             size=9, color=MUTED_, align="left")
    add_text(slide, f"{page_num} / {total}",
             Inches(11.5), Inches(7.15), Inches(1.2), Inches(0.25),
             size=9, color=MUTED_, align="right")

def fit_image(slide, img_path, box_left, box_top, box_w, box_h):
    """Fit image inside bounding box, preserve aspect ratio, center."""
    with Image.open(img_path) as im:
        iw, ih = im.size
    scale = min(box_w / iw, box_h / ih)
    w = int(iw * scale)
    h = int(ih * scale)
    left = box_left + (box_w - w) // 2
    top = box_top + (box_h - h) // 2
    return slide.shapes.add_picture(str(img_path), left, top, width=w, height=h)

print("PPTX helpers defined.")

PPTX helpers defined.


### Cell 9 — Slide builders

In [4]:
def slide_cover(prs):
    s = prs.slides.add_slide(prs.slide_layouts[6])
    add_text(s, "CASE STUDY", Inches(0.8), Inches(1.4), Inches(4), Inches(0.35),
             size=12, color=ACCENT, bold=True)
    add_text(s, "Olist E-commerce Analysis",
             Inches(0.8), Inches(1.9), Inches(11.7), Inches(1.3),
             size=48, bold=True, color=NEUTRAL_)
    add_text(s,
             "Delivery — not price, category, or payment — is the single driver of customer satisfaction.",
             Inches(0.8), Inches(3.4), Inches(11), Inches(1.4),
             size=20, color=NEUTRAL_, italic=True)
    add_multi_text(s, [
        {"text": AUTHOR,     "size": 14, "bold": True, "color": NEUTRAL_, "space_after": 2},
        {"text": GITHUB_URL, "size": 12, "color": ACCENT, "space_after": 2},
        {"text": DATE_STR,   "size": 12, "color": MUTED_},
    ], Inches(0.8), Inches(5.9), Inches(11), Inches(1.2))
    return s

def slide_thesis(prs):
    s = prs.slides.add_slide(prs.slide_layouts[6])
    add_text(s, "THESIS", Inches(0.8), Inches(1.4), Inches(4), Inches(0.35),
             size=12, color=ACCENT, bold=True)
    add_multi_text(s, [
        {"text": "Olist satisfaction is delivery-driven.",
         "size": 36, "bold": True, "color": NEUTRAL_, "space_after": 14},
        {"text": ("Price, category, and payment type each produce trivial or null effects "
                  "on review score — despite strong between-category price heterogeneity."),
         "size": 20, "color": NEUTRAL_, "space_after": 14},
        {"text": "The one operational lever is on-time percentage — not average delivery speed.",
         "size": 20, "color": ACCENT, "italic": True},
    ], Inches(0.8), Inches(2.3), Inches(11.7), Inches(4.5), anchor="top")
    return s

def slide_headline(prs, figure_path):
    s = prs.slides.add_slide(prs.slide_layouts[6])
    fit_image(s, figure_path,
              box_left=Inches(0.4), box_top=Inches(0.4),
              box_w=Inches(12.5),   box_h=Inches(6.55))
    return s

def slide_methodology(prs):
    s = prs.slides.add_slide(prs.slide_layouts[6])
    add_text(s, "METHODOLOGY", Inches(0.8), Inches(0.5), Inches(6), Inches(0.35),
             size=12, color=ACCENT, bold=True)
    add_text(s, "How the analysis was done",
             Inches(0.8), Inches(0.9), Inches(11), Inches(0.6),
             size=28, bold=True, color=NEUTRAL_)

    stats = [
        ("~96k",  "delivered orders",  "Jan 2017 – Aug 2018, 20 full months"),
        ("99.3%", "review coverage",   "Flat across status, month, delivery speed"),
        ("9",     "statistical tests", "Spearman, Kruskal-Wallis, Mann-Whitney, Welch's, χ², z"),
    ]
    col_w, y, gap = Inches(3.85), Inches(2.05), Inches(0.2)
    x = Inches(0.8)
    for i, (big, small, desc) in enumerate(stats):
        left = x + (col_w + gap) * i
        add_panel(s, left, y, col_w, Inches(1.9), fill=SOFT)
        add_text(s, big,   left, y + Inches(0.18), col_w, Inches(0.9),
                 size=44, bold=True, color=ACCENT, align="center")
        add_text(s, small, left, y + Inches(1.05), col_w, Inches(0.4),
                 size=13, bold=True, color=NEUTRAL_, align="center")
        add_text(s, desc,  left + Inches(0.2), y + Inches(1.45),
                 col_w - Inches(0.4), Inches(0.5),
                 size=10, color=MUTED_, align="center")

    add_text(s, "Key decisions", Inches(0.8), Inches(4.3), Inches(11), Inches(0.4),
             size=14, bold=True, color=NEUTRAL_)

    left_col = [
        ("Review score = ordinal.",   "Top-box (5★) and bottom-box (1★) rates instead of means."),
        ("Delivered-only filter.",    "Satisfaction tests restricted to completed deliveries."),
        ("Effect size > p-value.",    "At n≈96k, significance is near-universal."),
    ]
    right_col = [
        ("Bootstrap CIs on ρ.",       "2000 resamples, paired, percentile, seed=42."),
        ("Min-n filter, 100.",        "20 tiny categories dropped — 0.8% of sample."),
        ("Cohen effect thresholds.",  "Trivial <0.10, weak <0.30, moderate <0.50, strong ≥0.50."),
    ]
    def col(blocks, cleft, ctop, cwidth):
        entries = []
        for head, body in blocks:
            entries.append({"text": head, "size": 12, "bold": True, "color": NEUTRAL_, "space_after": 1})
            entries.append({"text": body, "size": 11, "color": MUTED_, "space_after": 8})
        add_multi_text(s, entries, cleft, ctop, cwidth, Inches(2.2))
    col(left_col,  Inches(0.8),  Inches(4.75), Inches(5.85))
    col(right_col, Inches(6.85), Inches(4.75), Inches(5.85))
    return s

def slide_limitations(prs):
    s = prs.slides.add_slide(prs.slide_layouts[6])
    add_text(s, "LIMITATIONS", Inches(0.8), Inches(0.5), Inches(6), Inches(0.35),
             size=12, color=ACCENT, bold=True)
    add_text(s, "What this analysis can and can't claim",
             Inches(0.8), Inches(0.9), Inches(11), Inches(0.6),
             size=28, bold=True, color=NEUTRAL_)
    items = [
        ("01", "Observational, not causal.",
         "The analysis identifies correlations and consistent mediation patterns. "
         "It does not rule out unmeasured confounders."),
        ("02", "Reviews dated before delivery (~8.5%) were retained, not filtered.",
         "These represent customers submitting dispute reviews while still awaiting late shipments — "
         "the most dissatisfied segment. Removing them would bias top-box rates upward and "
         "systematically understate the delivery-satisfaction relationship."),
        ("03", "Analytical window: Jan 2017 – Aug 2018.",
         "Excludes late 2016 (low coverage) and Sep 2018 onward (partial month). "
         "Black Friday 2017 retained; coverage verified flat."),
        ("04", "Minimum-n = 100 for category-level tests.",
         "Cleaner rank-statistics and Dunn's matrix at the cost of 0.8% sample coverage. "
         "Trade-off documented in notebook 03."),
    ]
    y, row_h = Inches(2.0), Inches(1.15)
    for i, (num, head, body) in enumerate(items):
        top = y + row_h * i
        add_text(s, num, Inches(0.8), top, Inches(0.9), Inches(1.1),
                 size=36, bold=True, color=MUTED_, align="left")
        add_multi_text(s, [
            {"text": head, "size": 14, "bold": True, "color": NEUTRAL_, "space_after": 2},
            {"text": body, "size": 11, "color": NEUTRAL_},
        ], Inches(1.9), top + Inches(0.05), Inches(10.8), row_h)
    return s

def slide_next_steps(prs):
    s = prs.slides.add_slide(prs.slide_layouts[6])
    add_text(s, "WHAT I'D DO NEXT", Inches(0.8), Inches(0.5), Inches(6), Inches(0.35),
             size=12, color=ACCENT, bold=True)
    add_text(s, "Three investigations this data opens up",
             Inches(0.8), Inches(0.9), Inches(11), Inches(0.6),
             size=28, bold=True, color=NEUTRAL_)
    cards = [
        ("office_furniture", "Retention dive",
         "The only category combining top-5 median price with bottom-5 satisfaction. "
         "Plausible mechanisms — bulk-shipping damage, assembly complexity, expectation-price "
         "mismatch — cannot be confirmed from order data alone. Would require review-text NLP "
         "or a targeted customer survey."),
        ("Rio de Janeiro", "Last-mile investigation",
         "RJ sits ~5pp above the delivery-time trend line across top-10 states. "
         "Same mean delivery as SC/GO/RS, but measurably worse satisfaction. "
         "Points to a carrier-handoff or final-mile problem, not a logistics-distance problem."),
        ("Voucher orders", "Composition audit",
         "Voucher completion runs 4.7pp below other payment methods at 1.6% of orders. "
         "Most likely explanation is customer-intent composition (promo-driven buyers), "
         "not payment-infrastructure failure — but the intent data would need to be joined."),
    ]
    card_w, card_h = Inches(3.85), Inches(4.2)
    x, y, gap = Inches(0.8), Inches(2.0), Inches(0.2)
    for i, (tag, title, body) in enumerate(cards):
        left = x + (card_w + gap) * i
        add_panel(s, left, y, card_w, card_h, fill=SOFT)
        add_text(s, tag,   left + Inches(0.3), y + Inches(0.25),
                 card_w - Inches(0.6), Inches(0.35), size=10, bold=True, color=ACCENT)
        add_text(s, title, left + Inches(0.3), y + Inches(0.6),
                 card_w - Inches(0.6), Inches(0.6), size=20, bold=True, color=NEUTRAL_)
        add_text(s, body,  left + Inches(0.3), y + Inches(1.3),
                 card_w - Inches(0.6), card_h - Inches(1.5),
                 size=12, color=NEUTRAL_)
    return s

def slide_thanks(prs):
    s = prs.slides.add_slide(prs.slide_layouts[6])
    add_text(s, "THANK YOU", Inches(0.8), Inches(1.5), Inches(6), Inches(0.35),
             size=12, color=ACCENT, bold=True)
    add_text(s, "Questions welcome.",
             Inches(0.8), Inches(2.0), Inches(11), Inches(1.1),
             size=40, bold=True, color=NEUTRAL_)
    add_text(s, "If a finding stuck — or didn't — I want to know which one.",
             Inches(0.8), Inches(3.2), Inches(11), Inches(0.7),
             size=18, color=NEUTRAL_, italic=True)
    rows = [("GitHub", GITHUB_URL), ("LinkedIn", LINKEDIN_URL), ("Email", EMAIL)]
    y0 = Inches(4.7)
    for i, (label, val) in enumerate(rows):
        top = y0 + Inches(0.45) * i
        add_text(s, label, Inches(0.8), top, Inches(1.4), Inches(0.4),
                 size=12, bold=True, color=MUTED_)
        add_text(s, val,   Inches(2.3), top, Inches(10), Inches(0.4),
                 size=13, color=NEUTRAL_)
    return s

def slide_appendix_a(prs):
    s = prs.slides.add_slide(prs.slide_layouts[6])
    add_text(s, "APPENDIX A", Inches(0.8), Inches(0.5), Inches(6), Inches(0.35),
             size=12, color=ACCENT, bold=True)
    add_text(s, "Full statistical results",
             Inches(0.8), Inches(0.9), Inches(11), Inches(0.6),
             size=24, bold=True, color=NEUTRAL_)

    headers = ["#", "Relationship", "Test", "Result & effect size", "Verdict"]
    rows = [
        ["Q1",   "price → delivery",        "Spearman ρ",                    "ρ = +0.102, CI [+0.095, +0.108]",      "weak"],
        ["§4-A", "delivery → review",       "Spearman ρ",                    "ρ = −0.235 — 30-day cliff",             "headline"],
        ["§4-B", "on-time → review",        "Mann-Whitney + 2-prop z",        "|r| = 0.55; OR = 12.2× [11.6, 12.8]",   "headline"],
        ["Q2",   "price → review",          "Spearman ρ",                    "ρ = −0.029 (mediation consistent)",     "trivial"],
        ["Q3",   "category → review",       "Kruskal-Wallis + Dunn's",        "η²_H = 0.008; 186/1275 pairs sig",      "trivial"],
        ["Q4a",  "payment → completion",    "Chi-square + Cramer's V",        "V = 0.035; voucher −4.7pp",             "trivial"],
        ["Q4b",  "payment → review",        "Kruskal-Wallis",                 "η²_H ≈ 0.000; p = 0.060",               "trivial"],
        ["§1",   "category → log₁₀(price)", "Welch's ANOVA + Games-Howell",   "ω² = 0.129; 899/1275 pairs sig",        "moderate"],
    ]
    col_widths_in = [0.7, 2.3, 2.7, 4.5, 1.8]
    total_w = sum(col_widths_in)
    left = Inches((13.333 - total_w) / 2)
    top = Inches(1.9)
    height = Inches(4.6)

    table_shape = s.shapes.add_table(len(rows) + 1, len(headers), left, top,
                                     Inches(total_w), height)
    tbl = table_shape.table
    for j, w in enumerate(col_widths_in):
        tbl.columns[j].width = Inches(w)

    for j, h in enumerate(headers):
        cell = tbl.cell(0, j)
        cell.text = ""
        cell.fill.solid(); cell.fill.fore_color.rgb = SOFT
        cell.margin_left = cell.margin_right = Inches(0.1)
        cell.margin_top = cell.margin_bottom = Inches(0.06)
        p = cell.text_frame.paragraphs[0]; p.alignment = PP_ALIGN.LEFT
        _set_run(p.add_run(), h, size=11, bold=True, color=NEUTRAL_)

    for i, row in enumerate(rows):
        for j, val in enumerate(row):
            cell = tbl.cell(i + 1, j)
            cell.text = ""
            cell.fill.solid(); cell.fill.fore_color.rgb = WHITE
            cell.margin_left = cell.margin_right = Inches(0.1)
            cell.margin_top = cell.margin_bottom = Inches(0.06)
            p = cell.text_frame.paragraphs[0]; p.alignment = PP_ALIGN.LEFT
            color, bold = NEUTRAL_, False
            if j == 4:
                if val == "headline":   color, bold = ACCENT, True
                elif val == "moderate": color, bold = NEUTRAL_, True
                elif val == "trivial":  color = MUTED_
            _set_run(p.add_run(), val, size=10, bold=bold, color=color)
    return s

def slide_appendix_fig(prs, label, title, figure_path):
    s = prs.slides.add_slide(prs.slide_layouts[6])
    add_text(s, label, Inches(0.8), Inches(0.5), Inches(6), Inches(0.35),
             size=12, color=ACCENT, bold=True)
    add_text(s, title, Inches(0.8), Inches(0.9), Inches(11.7), Inches(0.6),
             size=22, bold=True, color=NEUTRAL_)
    fit_image(s, figure_path,
              box_left=Inches(0.8), box_top=Inches(1.75),
              box_w=Inches(11.7),   box_h=Inches(5.15))
    return s

print("Slide builders defined.")

Slide builders defined.


### Cell 10 — Assemble the deck

In [5]:
prs = Presentation()
prs.slide_width = SLIDE_W
prs.slide_height = SLIDE_H

slides = []
slides.append(("cover",       slide_cover(prs)))
slides.append(("thesis",      slide_thesis(prs)))
slides.append(("headline_1",  slide_headline(prs, FIGURES_DIR / "slide_01_thirtyday_cliff.png")))
slides.append(("headline_2",  slide_headline(prs, FIGURES_DIR / "slide_02_ontime_vs_late.png")))
slides.append(("headline_3",  slide_headline(prs, FIGURES_DIR / "slide_03_thesis_contrast.png")))
slides.append(("headline_4",  slide_headline(prs, FIGURES_DIR / "slide_05_state_delivery_mediation.png")))
slides.append(("methodology", slide_methodology(prs)))
slides.append(("limitations", slide_limitations(prs)))
slides.append(("next_steps",  slide_next_steps(prs)))
slides.append(("thanks",      slide_thanks(prs)))
slides.append(("appx_a",      slide_appendix_a(prs)))
slides.append(("appx_b",      slide_appendix_fig(prs, "APPENDIX B",
                                                "office_furniture — the one category combining high price and high 1★ rate",
                                                FIGURES_DIR / "slide_04_office_furniture.png")))
slides.append(("appx_c",      slide_appendix_fig(prs, "APPENDIX C",
                                                "Q1 — Price vs delivery time (weak positive, ρ = +0.102)",
                                                FIGURES_DIR / "07_q1_price_vs_delivery.png")))
slides.append(("appx_d",      slide_appendix_fig(prs, "APPENDIX D",
                                                "§1 — Category explains price (ω² = 0.129, moderate)",
                                                FIGURES_DIR / "13_s1_category_vs_price.png")))

# Footer everywhere except cover & thanks (section slides)
skip_footer = {"cover", "thanks"}
total = len(slides)
for i, (name, slide) in enumerate(slides):
    if name in skip_footer:
        continue
    add_footer(slide, i + 1, total)

prs.save(OUT_PPTX)
print(f"Wrote {OUT_PPTX.relative_to(PROJECT_ROOT)}")
print(f"File size: {OUT_PPTX.stat().st_size / 1024:.0f} KB")
print(f"Slides: {total}")

Wrote reports/olist_deck.pptx
File size: 1182 KB
Slides: 14


### Cell 11 — Export to PDF

Two options:

1. **Manual (recommended on Windows):** open `reports/olist_deck.pptx` in PowerPoint → *File → Export → Create PDF/XPS* → save as `reports/olist_deck.pdf`.
2. **Automated (if LibreOffice installed):** the cell below attempts a headless conversion via `soffice`. Falls back to manual instructions if not found.

In [ ]:
import shutil, subprocess

soffice = shutil.which("soffice") or shutil.which("libreoffice")
if soffice:
    print(f"Found {soffice} — converting to PDF...")
    result = subprocess.run(
        [soffice, "--headless", "--convert-to", "pdf",
         "--outdir", str(OUT_PPTX.parent), str(OUT_PPTX)],
        capture_output=True, text=True, timeout=60
    )
    pdf_path = OUT_PPTX.with_suffix(".pdf")
    if pdf_path.exists():
        print(f"Wrote {pdf_path.relative_to(PROJECT_ROOT)}  ({pdf_path.stat().st_size / 1024:.0f} KB)")
    else:
        print("soffice ran but no PDF produced.")
        print("stdout:", result.stdout); print("stderr:", result.stderr)
else:
    print("LibreOffice (soffice) not found on PATH.")
    print()
    print("To export the PDF manually:")
    print(f"  1. Open {OUT_PPTX}")
    print("  2. File → Export → Create PDF/XPS")
    print(f"  3. Save as {OUT_PPTX.with_suffix('.pdf')}")

LibreOffice (soffice) not found on PATH.

To export the PDF manually:
  1. Open c:\Users\thinkpad\Desktop\olist-ecommerce-analysis\reports\olist_deck.pptx
  2. File → Export → Create PDF/XPS
  3. Save as c:\Users\thinkpad\Desktop\olist-ecommerce-analysis\reports\olist_deck.pdf
